In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_score,KFold
from sklearn.metrics import roc_auc_score , log_loss
import warnings
warnings.filterwarnings("ignore")
from sklearn.exceptions import ConvergenceWarning
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import Pipeline
import os


In [3]:
df = pd.read_csv("F:\MACHINE LEARNING\SOMETHING\Cases\Medical_Cost_Personal\insurance.csv")
X,y = df.drop('charges',axis=1),df['charges']

In [4]:
ohe = OneHotEncoder(sparse_output=False,drop='first').set_output(transform='pandas')
ct = make_column_transformer((ohe,make_column_selector(dtype_include='object')),
                            ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False)

xgb = XGBRegressor(random_state=26)
pipe = Pipeline([('OHE',ct),('GBM',xgb)])
kfold = KFold(n_splits=5,random_state=26,shuffle=True)

In [ ]:
# from sklearn.model_selection import GridSearchCV
# params = {
#     'learning_rate' : np.linspace(0.01,0.9,10),
#     'max_depth' : [None,2,3,4],
#     'n_estimators' : [25,50,100]
# }
# gcv = GridSearchCV(pipe, param_grid=params, cv=kfold)
# gcv.fit(X,y)
# print("Best param : ", gcv.best_params_)
# print("Best score : ", gcv.best_score_)

### The above code throws error because of pipeline
### Because of pipeline the parameter name changes so we have to give same name in our params

In [5]:
pipe.get_params()

{'memory': None,
 'steps': [('OHE',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000001F05F3FB380>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000001F05F46D450>)],
                     verbose_feature_names_out=False)),
  ('GBM',
   XGBRegressor(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_bynode=None,
                colsample_bytree=None, device=None, early_stopping_rounds=None,
                enable_categorical=True, eval_metric=None, feature_types=None,
                feature_weights=None, gamma=None, grow_policy=None,
                importance_type=None,

In [6]:
from sklearn.model_selection import GridSearchCV
params = {
    'GBM__learning_rate' : np.linspace(0.01,0.9,10),
    'GBM__max_depth' : [None,2,3,4],
    'GBM__n_estimators' : [25,50,100]
}
gcv = GridSearchCV(pipe, param_grid=params, cv=kfold)
gcv.fit(X,y)
print("Best param : ", gcv.best_params_)
print("Best score : ", gcv.best_score_)

Best param :  {'GBM__learning_rate': np.float64(0.10888888888888888), 'GBM__max_depth': 3, 'GBM__n_estimators': 50}
Best score :  0.8632949136193883


In [ ]:
params = {
    'GBM__learning_rate' : np.linspace(0.01,0.9,10),
    'GBM__max_depth' : [None,2,3,4],
    'GBM__n_estimators' : [25,50,100]
}
gcv = GridSearchCV(pipe, param_grid=params, cv=kfold,scoring='r2',verbose=1)
gcv.fit(X,y)

In [ ]:
gcv = GridSearchCV(pipe, param_grid=params, cv=kfold,scoring='r2',verbose=2)
gcv.fit(X,y)